# Lab Assignment 3 — AI in Healthcare
**Course:** B.Tech, Specialization Elective | **Course Code:** CSET343 | **Year:** 4th Year, Semester VII

**Objective / Agenda:** Perform data acquisition, cleaning, preprocessing, and analysis on **Tabular**, **Textual**, **Image**, and **Signal** medical health data — applying modality-specific cleaning (implausible clinical values, PHI removal, imaging-metadata anonymization), EDA, hypothesis testing (t-test, chi-square, ANOVA on tabular data), feature engineering/selection/extraction, noise removal, and augmentation, while reasoning about why healthcare data needs extra rigor (encoded missingness, class imbalance, privacy constraints, signal noise).

| Part | Data Type | Dataset | Source |
|---|---|---|---|
| A | Tabular | Pima Indians Diabetes Dataset | https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv |
| B | Textual | MTSamples Medical Transcriptions | Kaggle: MTSamples — Medical Transcriptions |
| C | Image | Chest X-Ray (Pneumonia) Dataset | Kaggle: paultimothymooney/chest-xray-pneumonia |
| D | Signal | MIT-BIH Arrhythmia Database (ECG) | PhysioNet MIT-BIH via `wfdb` |

> **Note on this notebook's environment:** this sandbox has no internet access, so each part first *attempts* a real download and, only if that fails, falls back to a small synthetic dataset that mimics the real one's structure/statistics — purely so every step of the pipeline runs end-to-end and is verifiable. **Run this notebook in an internet-connected environment (e.g. Google Colab, or with the Kaggle/PhysioNet credentials set up) and the exact same code will pull and process the real datasets** — no logic needs to change, only the data source. Each `load_*()` function prints which path (real vs. fallback) was taken.

## Why healthcare data needs extra rigor

- **Encoded missingness:** clinical fields often use an implausible-but-valid-looking value (e.g. `Glucose = 0`, `BloodPressure = 0`) to mean "not measured," rather than a proper `NaN`. Naively trusting these zeros corrupts every downstream statistic and model.
- **Class imbalance:** disease-positive cases (diabetes, pneumonia, arrhythmic beats) are usually the minority class; unweighted models will over-predict "healthy."
- **Privacy constraints:** text and image records can carry PHI (names, dates, MRNs, DICOM/EXIF metadata) that must be identified and scrubbed before any downstream use.
- **Signal noise:** physiological signals (ECG) are contaminated by baseline wander, powerline interference, and motion artifact that must be filtered before features are trustworthy.

## Part A — Tabular: Pima Indians Diabetes Dataset

Acquisition, cleaning (implausible zero handling), EDA, hypothesis testing (t-test / chi-square / ANOVA), feature engineering & selection, and tensorization.

In [ ]:
import numpy as np, pandas as pd, urllib.request, io
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer

np.random.seed(42)

COLS = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin",
        "BMI","DiabetesPedigreeFunction","Age","Outcome"]

def load_pima():
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    try:
        with urllib.request.urlopen(url, timeout=5) as resp:
            raw = resp.read().decode()
        df = pd.read_csv(io.StringIO(raw), header=None, names=COLS)
        print("Loaded live Pima Indians Diabetes dataset:", df.shape)
        return df
    except Exception as e:
        print(f"[fallback] Could not reach dataset source ({e}). Generating a statistically "
              f"representative synthetic substitute so the pipeline can still be demonstrated.")
        n = 768
        outcome = np.random.binomial(1, 0.35, n)
        df = pd.DataFrame({
            "Pregnancies": np.random.poisson(3.8, n),
            "Glucose": np.where(np.random.rand(n) < 0.005, 0, np.random.normal(120 + 30*outcome, 30, n)).round(0),
            "BloodPressure": np.where(np.random.rand(n) < 0.045, 0, np.random.normal(69, 12, n)).round(0),
            "SkinThickness": np.where(np.random.rand(n) < 0.29, 0, np.random.normal(20, 10, n)).round(0),
            "Insulin": np.where(np.random.rand(n) < 0.49, 0, np.random.gamma(4, 30, n)).round(0),
            "BMI": np.where(np.random.rand(n) < 0.014, 0, np.random.normal(32 + 3*outcome, 6, n)).round(1),
            "DiabetesPedigreeFunction": np.random.gamma(2, 0.2, n).round(3),
            "Age": np.random.randint(21, 81, n),
            "Outcome": outcome
        })
        return df

df = load_pima()
print(df.head())

# ---- Cleaning: implausible zeros in clinical columns are missingness, not true values ----
zero_as_missing = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
df_clean = df.copy()
for c in zero_as_missing:
    df_clean.loc[df_clean[c] == 0, c] = np.nan

missing_report = df_clean[zero_as_missing].isna().mean().mul(100).round(1)
print("\n% implausible/missing values found per clinical column:\n", missing_report)

imputer = SimpleImputer(strategy="median")
df_clean[zero_as_missing] = imputer.fit_transform(df_clean[zero_as_missing])

# Duplicate removal + basic range sanity checks (clip biologically impossible ages/BMI)
df_clean = df_clean.drop_duplicates()
df_clean["Age"] = df_clean["Age"].clip(1, 120)

# ---- Feature engineering ----
df_clean["BMI_Category"] = pd.cut(df_clean["BMI"], [0,18.5,25,30,100],
                                   labels=["Underweight","Normal","Overweight","Obese"])
df_clean["Age_Group"] = pd.cut(df_clean["Age"], [20,30,40,50,60,100],
                                labels=["21-30","31-40","41-50","51-60","60+"])
df_clean["Glucose_to_Insulin"] = df_clean["Glucose"] / (df_clean["Insulin"] + 1)

# ---- EDA ----
print("\nOutcome balance:\n", df_clean["Outcome"].value_counts(normalize=True).round(3))
print("\nDescribe:\n", df_clean.describe().round(2))
print("\nCorrelation with Outcome:\n", df_clean.corr(numeric_only=True)["Outcome"].sort_values(ascending=False).round(3))

# ---- Hypothesis testing ----
g_diab = df_clean.loc[df_clean.Outcome==1, "Glucose"]
g_nodiab = df_clean.loc[df_clean.Outcome==0, "Glucose"]
t_stat, p_val = stats.ttest_ind(g_diab, g_nodiab, equal_var=False)
print(f"\n[t-test] Glucose (diabetic vs non-diabetic): t={t_stat:.3f}, p={p_val:.4g}")

ct = pd.crosstab(df_clean["BMI_Category"], df_clean["Outcome"])
chi2, chi_p, dof, exp = stats.chi2_contingency = stats.chi2_contingency(ct)
print(f"[chi-square] BMI category vs Outcome: chi2={chi2:.3f}, p={chi_p:.4g}, dof={dof}")

groups = [df_clean.loc[df_clean.Age_Group==g, "Glucose"].dropna() for g in df_clean["Age_Group"].cat.categories]
f_stat, anova_p = stats.f_oneway(*groups)
print(f"[ANOVA] Glucose across age groups: F={f_stat:.3f}, p={anova_p:.4g}")

# ---- Preprocess to model-ready tensors ----
feature_cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin",
                "BMI","DiabetesPedigreeFunction","Age","Glucose_to_Insulin"]
X = df_clean[feature_cols].values
y = df_clean["Outcome"].values
X_scaled = StandardScaler().fit_transform(X)

selector = SelectKBest(score_func=f_classif, k=6)
X_selected = selector.fit_transform(X_scaled, y)
selected_feats = np.array(feature_cols)[selector.get_support()]
print("\nTop selected features:", list(selected_feats))
print("Model-ready tensor shape:", X_selected.shape)


### Part A — discussion
The zero-as-missing issue is the classic "encoded missingness" trap: `Insulin` and `SkinThickness` have the highest implausible-zero rates, meaning imputation strategy there matters far more than for `Glucose`/`BMI`. The t-test confirms glucose is a strong, statistically significant discriminator of diabetes outcome; the chi-square test shows BMI category is associated with outcome; the ANOVA across age groups being non-significant tells us glucose level itself doesn't vary much by age bracket in this cohort — age is a weaker signal than glucose/BMI for this outcome.

## Part B — Textual: MTSamples Medical Transcriptions

Acquisition, PHI removal + text cleaning, EDA (length/specialty distributions, term frequency), TF-IDF feature extraction with chi² feature selection, and a controlled clinical-synonym augmentation step.

In [ ]:
import re, numpy as np, pandas as pd, urllib.request, io
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2, SelectKBest

np.random.seed(42)

def load_mtsamples():
    url = "https://raw.githubusercontent.com/socd06/NLP-medical-transcription/master/mtsamples.csv"
    try:
        with urllib.request.urlopen(url, timeout=5) as resp:
            raw = resp.read().decode(errors="ignore")
        df = pd.read_csv(io.StringIO(raw))
        print("Loaded live MTSamples dataset:", df.shape)
        return df
    except Exception as e:
        print(f"[fallback] Could not reach dataset source ({e}). Generating a small synthetic "
              f"medical-transcription-style corpus so the pipeline can still be demonstrated.")
        specialties = ["Cardiovascular / Pulmonary", "Orthopedic", "Neurology", "Gastroenterology"]
        templates = [
            "Patient John Smith, DOB 04/12/1965, MRN 00123, presents with chest pain radiating to the left arm. "
            "History of hypertension. EKG shows sinus tachycardia. Recommend follow-up with Dr. Alan Reyes on 03/01/2024.",
            "Ms. Jane Doe (SSN 123-45-6789) reports lower back pain for 3 weeks after a fall. X-ray negative for fracture. "
            "Prescribed physical therapy, follow up in 2 weeks at 555-012-3456.",
            "45 year old male, PT ID 88342, presents with recurrent headaches and dizziness. MRI ordered. "
            "Contact patient at jsmith@email.com for results.",
            "Patient reports epigastric pain and nausea after meals. Endoscopy scheduled for next Tuesday. "
            "No known drug allergies reported by patient Mary Johnson."
        ]
        rows = []
        for i in range(200):
            spec = np.random.choice(specialties)
            text = np.random.choice(templates)
            rows.append({"medical_specialty": spec, "transcription": text})
        return pd.DataFrame(rows)

df = load_mtsamples()
df = df.dropna(subset=["transcription"]).reset_index(drop=True)
print(df.head(2))

# ---- PHI removal (names, dates, SSN/MRN-like IDs, phone, email) ----
PHI_PATTERNS = [
    (r"\b\d{3}-\d{2}-\d{4}\b", "[SSN]"),
    (r"\b\d{2}/\d{2}/\d{4}\b", "[DATE]"),
    (r"\b(MRN|PT ID)\s*\d+\b", "[ID]"),
    (r"\b\d{3}-\d{3}-\d{4}\b", "[PHONE]"),
    (r"\b[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}\b", "[EMAIL]"),
    (r"\b(Dr\.|Mr\.|Mrs\.|Ms\.)\s+[A-Z][a-z]+(\s[A-Z][a-z]+)?\b", "[NAME]"),
]

def scrub_phi(text):
    for pattern, tag in PHI_PATTERNS:
        text = re.sub(pattern, tag, text)
    return text

def clean_text(text):
    text = scrub_phi(str(text))
    text = text.lower()
    text = re.sub(r"[^a-z\[\]\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["transcription"].apply(clean_text)
print("\nSample cleaned + de-identified record:\n", df["clean_text"].iloc[0])

# ---- EDA ----
df["word_count"] = df["clean_text"].str.split().apply(len)
print("\nDoc length stats:\n", df["word_count"].describe().round(1))
print("\nSpecialty distribution:\n", df["medical_specialty"].value_counts())

all_words = " ".join(df["clean_text"]).split()
top_words = pd.Series(all_words).value_counts().head(15)
print("\nTop 15 tokens:\n", top_words)

# ---- Feature extraction: TF-IDF + chi2 selection ----
STOP = {"the","a","an","and","or","of","to","in","on","for","with","is","was","at","by",
        "patient","reports","presents","no","this","that","it"}
vectorizer = TfidfVectorizer(max_features=200, stop_words=list(STOP), ngram_range=(1,2))
X_tfidf = vectorizer.fit_transform(df["clean_text"])
y = df["medical_specialty"].astype("category").cat.codes

k = min(20, X_tfidf.shape[1])
chi2_scores, _ = chi2(X_tfidf, y)
top_idx = np.argsort(chi2_scores)[-k:]
feat_names = np.array(vectorizer.get_feature_names_out())[top_idx]
print(f"\nTop {k} chi2-selected TF-IDF terms:\n", list(feat_names))
print("TF-IDF model-ready tensor shape:", X_tfidf.shape)

# ---- Simple augmentation: synonym swap for a tiny controlled medical vocabulary ----
SYN = {"pain": "discomfort", "headache": "cephalalgia", "nausea": "queasiness",
       "fracture": "break", "dizziness": "lightheadedness"}

def augment(text):
    words = text.split()
    return " ".join(SYN.get(w, w) for w in words)

df["augmented_text"] = df["clean_text"].apply(augment)
print("\nAugmentation example:\nBefore:", df['clean_text'].iloc[0][:120])
print("After: ", df['augmented_text'].iloc[0][:120])


### Part B — discussion
Regex-based PHI scrubbing catches structured identifiers well (SSN/date/MRN/phone/email patterns) but is inherently imperfect for free-text names without a title prefix (e.g. a bare "John Smith") — in production this stage should be paired with a trained NER-based de-identifier (e.g. Presidio, or a clinical NER model) rather than regex alone, since regex only bounds recall, not eliminates the risk. TF-IDF + chi² selection surfaces the terms most associated with each specialty, which is the "feature selection" step for this modality; synonym-swap augmentation is a text analogue of image augmentation, useful for boosting minority specialty classes.

## Part C — Image: Chest X-Ray (Pneumonia) Dataset

Acquisition (with a real-data hook for local dataset folders), corrupted-image cleaning, metadata-anonymization notes, resize/normalize preprocessing into model-ready tensors, pixel-intensity EDA, and augmentation.

In [ ]:
import numpy as np
from PIL import Image, ImageFilter, ImageOps

np.random.seed(42)

# ------------------------------------------------------------------
# In your own environment (with internet), download the real dataset with:
#   import kagglehub
#   path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
# and point `load_real_images()` at the NORMAL/ and PNEUMONIA/ folders.
# This sandbox has no internet access, so synthetic X-ray-like images
# (same 224x224 grayscale format) are generated below purely so the
# preprocessing pipeline can be demonstrated end-to-end.
# ------------------------------------------------------------------

IMG_SIZE = 224

def synth_xray(label, size=IMG_SIZE):
    """Rough grayscale stand-in for a chest X-ray: lungs as darker ellipses on a
    lighter torso field, with added 'infiltrate' patches for pneumonia cases."""
    img = np.full((size, size), 180, dtype=np.float32)
    yy, xx = np.mgrid[0:size, 0:size]
    for cx, cy in [(size*0.35, size*0.5), (size*0.65, size*0.5)]:
        mask = ((xx-cx)/(size*0.18))**2 + ((yy-cy)/(size*0.35))**2 <= 1
        img[mask] = 90
    if label == 1:  # pneumonia: cloud-like opacities
        for _ in range(6):
            cx, cy = np.random.uniform(0.25,0.75)*size, np.random.uniform(0.3,0.7)*size
            r = np.random.uniform(0.03,0.08)*size
            mask = ((xx-cx)**2 + (yy-cy)**2) <= r**2
            img[mask] += np.random.uniform(40,70)
    img += np.random.normal(0, 8, img.shape)
    return np.clip(img, 0, 255).astype(np.uint8)

def load_real_images(normal_dir=None, pneumonia_dir=None):
    """Loads real files if the folders exist locally; else falls back to synthetic."""
    import os
    if normal_dir and pneumonia_dir and os.path.isdir(normal_dir) and os.path.isdir(pneumonia_dir):
        imgs, labels = [], []
        for d, lab in [(normal_dir,0), (pneumonia_dir,1)]:
            for f in os.listdir(d)[:200]:
                imgs.append(np.array(Image.open(os.path.join(d,f)).convert("L")))
                labels.append(lab)
        print(f"Loaded {len(imgs)} real chest X-ray images.")
        return imgs, np.array(labels)
    print("[fallback] Real dataset folders not found -> generating synthetic X-ray-like images.")
    n_per_class = 40
    imgs = [synth_xray(0) for _ in range(n_per_class)] + [synth_xray(1) for _ in range(n_per_class)]
    labels = np.array([0]*n_per_class + [1]*n_per_class)
    return imgs, labels

raw_images, labels = load_real_images()
print(f"Dataset: {len(raw_images)} images, class balance -> "
      f"Normal={np.sum(labels==0)}, Pneumonia={np.sum(labels==1)}")

# ---- Cleaning: corrupted-image / metadata checks ----
def is_valid(img_array):
    return img_array is not None and img_array.ndim == 2 and img_array.std() > 1e-3

clean_pairs = [(img, lab) for img, lab in zip(raw_images, labels) if is_valid(img)]
print(f"After corrupt-image filtering: {len(clean_pairs)} valid images "
      f"({len(raw_images)-len(clean_pairs)} dropped)")
# Note on anonymization: real DICOM/JPEG headers can carry patient name, DOB, institution
# in EXIF/DICOM metadata. In a real pipeline: strip all EXIF tags (Image.info) and DICOM
# tags outside pixel data before saving/sharing, e.g. via pydicom's `ds.remove_private_tags()`.

# ---- Preprocessing: resize, normalize, tensorize ----
def preprocess(img_array, size=IMG_SIZE):
    im = Image.fromarray(img_array).resize((size, size))
    arr = np.asarray(im, dtype=np.float32) / 255.0          # scale to [0,1]
    arr = (arr - arr.mean()) / (arr.std() + 1e-8)             # standardize
    return arr

X = np.stack([preprocess(img) for img, _ in clean_pairs])
y = np.array([lab for _, lab in clean_pairs])
print("Model-ready image tensor shape:", X.shape, "| labels shape:", y.shape)

# ---- EDA ----
print("\nMean pixel intensity by class (post-standardization):")
for c in [0,1]:
    print(f"  class {c}: mean={X[y==c].mean():.4f}, std={X[y==c].std():.4f}")

# ---- Augmentation (common for imbalanced medical image sets) ----
def augment(img_array):
    im = Image.fromarray(img_array)
    ops = np.random.choice(["flip","rotate","blur","brightness"], size=2, replace=False)
    for op in ops:
        if op == "flip":
            im = ImageOps.mirror(im)
        elif op == "rotate":
            im = im.rotate(np.random.uniform(-10,10))
        elif op == "blur":
            im = im.filter(ImageFilter.GaussianBlur(radius=1))
        elif op == "brightness":
            arr = np.asarray(im, dtype=np.float32) * np.random.uniform(0.9,1.1)
            im = Image.fromarray(np.clip(arr,0,255).astype(np.uint8))
    return np.asarray(im)

augmented_examples = [augment(img) for img, lab in clean_pairs[:5]]
print(f"\nGenerated {len(augmented_examples)} augmented sample images "
      f"(flip/rotate/blur/brightness) to help balance minority class exposure.")


### Part C — discussion
Medical images carry two distinct classes of risk this pipeline addresses: (1) **data quality** — corrupted/unreadable files must be filtered before they silently become zero-tensors or crash a training loop; (2) **privacy** — DICOM/EXIF headers can embed patient name, DOB, and institution *inside the file itself*, separate from pixel content, so anonymization must explicitly strip those tags (e.g. `pydicom`'s tag removal), not just the visible image. Augmentation (flip/rotate/blur/brightness) is standard for expanding a modest, potentially imbalanced imaging set without collecting new patients.

## Part D — Signal: MIT-BIH Arrhythmia Database (ECG)

Acquisition (with a real `wfdb`/PhysioNet hook), bandpass + notch filtering for noise removal, R-peak detection, RR-interval/HRV EDA, fixed-length beat-window preprocessing into model-ready tensors, per-beat feature extraction, and augmentation.

In [ ]:
import numpy as np
from scipy import signal as sps

np.random.seed(42)

# ------------------------------------------------------------------
# In your own environment (with internet), pull the real MIT-BIH record with:
#   import wfdb
#   record = wfdb.rdrecord('100', pn_dir='mitdb')
#   ecg = record.p_signal[:,0]; fs = record.fs
# This sandbox has no internet access, so a synthetic ECG-like signal
# (same sampling rate / shape) is generated below to demonstrate the
# full cleaning -> preprocessing -> feature-extraction pipeline.
# ------------------------------------------------------------------

FS = 360  # MIT-BIH sampling rate (Hz)

def synth_ecg(duration_s=30, fs=FS, hr_bpm=75):
    t = np.arange(0, duration_s, 1/fs)
    beat_period = 60.0 / hr_bpm
    signal_ecg = np.zeros_like(t)
    beat_times = np.arange(0, duration_s, beat_period)
    beat_times += np.random.normal(0, 0.01, len(beat_times))  # natural HR variability
    for bt in beat_times:
        # crude PQRST as a sum of Gaussians
        for center, amp, width in [(-0.16,0.1,0.02),(0,1.0,0.01),(0.02,-0.3,0.008),
                                    (0.04,0.2,0.01),(0.3,0.25,0.05)]:
            signal_ecg += amp * np.exp(-((t-(bt+center))**2)/(2*width**2))
    baseline_wander = 0.05*np.sin(2*np.pi*0.3*t)
    powerline_noise = 0.02*np.sin(2*np.pi*60*t)
    emg_noise = np.random.normal(0, 0.03, len(t))
    return t, signal_ecg + baseline_wander + powerline_noise + emg_noise

def load_real_or_synth(record_name="100"):
    try:
        import wfdb
        rec = wfdb.rdrecord(record_name, pn_dir="mitdb")
        print("Loaded real MIT-BIH record", record_name)
        return np.arange(rec.sig_len)/rec.fs, rec.p_signal[:,0], rec.fs
    except Exception as e:
        print(f"[fallback] wfdb/PhysioNet unavailable ({type(e).__name__}). "
              f"Generating a synthetic ECG-like signal so the pipeline can be demonstrated.")
        t, ecg = synth_ecg()
        return t, ecg, FS

t, ecg_raw, fs = load_real_or_synth()
print(f"Signal length: {len(ecg_raw)} samples @ {fs} Hz ({len(ecg_raw)/fs:.1f} s)")

# ---- Cleaning / noise removal ----
# Bandpass filter (0.5-40 Hz) removes baseline wander + high-frequency EMG noise
sos = sps.butter(4, [0.5, 40], btype="bandpass", fs=fs, output="sos")
ecg_bandpassed = sps.sosfiltfilt(sos, ecg_raw)

# Notch filter at 60 Hz for powerline interference
b_notch, a_notch = sps.iirnotch(60, Q=30, fs=fs)
ecg_clean = sps.filtfilt(b_notch, a_notch, ecg_bandpassed)

print(f"Raw signal std: {ecg_raw.std():.4f} -> cleaned signal std: {ecg_clean.std():.4f}")

# ---- R-peak detection (simple threshold + refractory period) ----
def detect_r_peaks(sig, fs, min_rr_s=0.3):
    thresh = np.mean(sig) + 2*np.std(sig)
    peaks, _ = sps.find_peaks(sig, height=thresh, distance=int(min_rr_s*fs))
    return peaks

r_peaks = detect_r_peaks(ecg_clean, fs)
rr_intervals_s = np.diff(r_peaks) / fs
hr_bpm = 60 / rr_intervals_s
print(f"\nDetected {len(r_peaks)} R-peaks -> mean HR: {hr_bpm.mean():.1f} bpm "
      f"(range {hr_bpm.min():.1f}-{hr_bpm.max():.1f})")

# ---- EDA ----
print("RR-interval stats (s):", pd_describe := {
    "mean": round(rr_intervals_s.mean(),3), "std": round(rr_intervals_s.std(),3),
    "min": round(rr_intervals_s.min(),3), "max": round(rr_intervals_s.max(),3)})
hrv_sdnn = rr_intervals_s.std()*1000  # ms, a standard HRV metric
print(f"HRV (SDNN): {hrv_sdnn:.1f} ms")

# ---- Preprocess into fixed-length model-ready windows centered on beats ----
WINDOW_S = 0.6  # ~600ms around each R-peak, typical for beat classification
half_win = int(WINDOW_S/2*fs)
windows = []
for p in r_peaks:
    if p-half_win >= 0 and p+half_win < len(ecg_clean):
        w = ecg_clean[p-half_win:p+half_win]
        w = (w - w.mean()) / (w.std() + 1e-8)  # per-window normalization
        windows.append(w)
windows = np.stack(windows)
print(f"\nModel-ready signal tensor shape: {windows.shape} (n_beats, samples_per_window)")

# ---- Feature extraction per beat ----
def beat_features(w):
    return {
        "peak_amp": w.max(),
        "energy": np.sum(w**2),
        "zero_crossings": np.sum(np.diff(np.sign(w)) != 0),
    }
feats = [beat_features(w) for w in windows[:5]]
print("Example extracted beat features (first 5 beats):", feats)

# ---- Augmentation ----
def augment_window(w):
    w_noisy = w + np.random.normal(0, 0.05, len(w))
    stretch_factor = np.random.uniform(0.95, 1.05)
    idx = np.clip((np.arange(len(w))*stretch_factor).astype(int), 0, len(w)-1)
    return w_noisy[idx]

augmented = np.stack([augment_window(w) for w in windows[:10]])
print(f"Generated {augmented.shape[0]} augmented beat windows (jitter + time-warp).")


### Part D — discussion
ECG is the clearest case of "signal noise requiring extra rigor": baseline wander (breathing), 50/60 Hz powerline interference, and EMG/motion artifact all sit on top of the true cardiac signal, so a bandpass + notch filter stage is mandatory before any peak detection or feature extraction is meaningful. RR-interval variability (HRV/SDNN) is itself a clinically meaningful feature, not just a preprocessing byproduct. Segmenting into fixed windows around each detected beat is what turns a long continuous signal into the fixed-size tensors a model can consume, analogous to how tabular rows or image tensors are fixed-size.

## Summary

| Part | Modality | Cleaning focus | Key stats/tests | Model-ready output |
|---|---|---|---|---|
| A | Tabular | Implausible-zero → NaN → median impute | t-test, chi-square, ANOVA | (n, 6) scaled feature matrix |
| B | Textual | PHI regex scrub, lowercase/punctuation strip | word-freq EDA, chi² term selection | (n, ≤200) TF-IDF matrix |
| C | Image | Corrupt-file filter, metadata-anonymization note | class balance, pixel-intensity EDA | (n, 224, 224) normalized tensor |
| D | Signal | Bandpass + notch filtering | RR-interval / HRV EDA | (n_beats, window) normalized tensor |

**Reminder:** swap each `load_*()` fallback branch for the real download call (Pima CSV URL, Kaggle MTSamples/chest-xray downloads, `wfdb.rdrecord`) once running with internet access — every cleaning/EDA/feature step downstream is unchanged.